In [2]:
from dotenv import load_dotenv
import os 
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials


In [3]:
load_dotenv()

sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=os.getenv("SPOTIFY_CLIENT_ID"),
        client_secret=os.getenv("SPOTIFY_CLIENT_SECRET")
    )
)

In [4]:
results = sp.search(q="BTS", type="artist", limit=10)

artist = next(
    item for item in results["artists"]["items"]
    if item["name"].casefold() == "bts"
)

artist_id = artist["id"]
print(artist_id)


3Nrfpe0tUJi4K4DXYWgMUX


In [5]:
artist_id = "3Nrfpe0tUJi4K4DXYWgMUX"

albums = sp.artist_albums(
    artist_id,
    album_type="album",
    limit=10
)

for album in albums["items"]:
    print(album["name"], "-", album["id"])

KEEP SWIMMING - 6iPjmGZeonxBZ9r7Cjkezq
ARIRANG - 3ukkRHDHbN8tNRPKsGZR1h
PERMISSION TO DANCE ON STAGE - LIVE - 7xLhqoM0ilHRKUg2irVSDI
Proof - 6al2VdKbb6FIz9d7lU7WRB
BE - 6nYfHQnvkvOTNHnOhDT3sr
MAP OF THE SOUL : 7 ~ THE JOURNEY ~ - 1nScVw87kRJiT2bg2Kswhp
MAP OF THE SOUL : 7 - 5W1XY5ucNATjTULERvXx9j
MAP OF THE SOUL : PERSONA - 2KqlAl1Kl5fZvbFgJ0qFB6
Love Yourself 結 'Answer' - 43wFM1HquliY3iwKWzPN4y
Love Yourself 轉 'Tear' - 4NIqCxqP9o8Tp6tGLBqd8O


In [6]:
album_id = "3ukkRHDHbN8tNRPKsGZR1h"
tracks = sp.album_tracks(album_id)

for track in tracks["items"]:
    print(track["name"])

Body to Body
Hooligan
Aliens
FYA
2.0
No. 29
SWIM
Merry Go Round
NORMAL
Like Animals
they don’t know ’bout us
One More Night
Please
Into the Sun


In [7]:
# Instead of manually copying album IDs, loop through every album automatically:

albums = sp.artist_albums(
    artist_id,
    album_type="album",
    limit=10
)

for album in albums["items"]:
    print(f"\nAlbum: {album["name"]}")

    tracks = sp.album_tracks(album["id"])

    for track in tracks["items"]:
        print(f"  - {track['name']}")


Album: KEEP SWIMMING
  - SWIM
  - SWIM with RM (Chill Hip Hop Remix)
  - SWIM with Jin (Alternative Rock Remix)
  - SWIM with SUGA (Melodic Techno Remix)
  - SWIM with j-hope (Afrobeat Remix)
  - SWIM with Jimin (Slow Jam R&B Remix)
  - SWIM with V (Electronic Remix)
  - SWIM with Jung Kook (Acoustic Lofi Remix)
  - SWIM (Instrumental)

Album: ARIRANG
  - Body to Body
  - Hooligan
  - Aliens
  - FYA
  - 2.0
  - No. 29
  - SWIM
  - Merry Go Round
  - NORMAL
  - Like Animals
  - they don’t know ’bout us
  - One More Night
  - Please
  - Into the Sun

Album: PERMISSION TO DANCE ON STAGE - LIVE
  - ON - Live
  - Burning Up (FIRE) - Live
  - Dope - Live
  - DNA - Live
  - Blue & Grey - Live
  - Black Swan - Live
  - Blood Sweat & Tears - Live
  - FAKE LOVE - Live
  - Life Goes On - Live
  - Boy With Luv (Feat. Halsey) - Live
  - Dynamite - Live
  - Butter - Live
  - Telepathy - Live
  - Outro : Wings - Live
  - Stay - Live
  - So What - Live
  - IDOL - Live
  - Airplane pt.2 - Live
  - Sil

In [8]:
os.makedirs("../data/raw", exist_ok=True)

album_rows = []
track_rows = []

artist_id = "3Nrfpe0tUJi4K4DXYWgMUX"

albums = sp.artist_albums(
    artist_id,
    include_groups="album,single",
    limit=10
)

all_albums = albums["items"]

while albums["next"]:
    albums = sp.next(albums)
    all_albums.extend(albums["items"])

for album in all_albums:
    album_id = album["id"]

    album_rows.append({
        "album_id": album_id,
        "album_name": album["name"],
        "album_type": album["album_type"],
        "release_date": album["release_date"],
        "total_tracks": album["total_tracks"],
        "spotify_url": album["external_urls"]["spotify"],
        "image_url": album["images"][0]["url"] if album["images"] else None
    })

    tracks = sp.album_tracks(album_id)

    for track in tracks["items"]:
        track_rows.append({
            "track_id": track["id"],
            "track_name": track["name"],
            "album_id": album_id,
            "album_name": album["name"],
            "track_number": track["track_number"],
            "duration_ms": track["duration_ms"],
            "explicit": track["explicit"],
            "spotify_url": track["external_urls"]["spotify"]
        })

albums_df = pd.DataFrame(album_rows)
tracks_df = pd.DataFrame(track_rows)

albums_df.to_csv("../data/raw/spotify_albums.csv", index=False)
tracks_df.to_csv("../data/raw/spotify_tracks.csv", index=False)

print("Saved albums:", albums_df.shape)
print("Saved tracks:", tracks_df.shape)

Saved albums: (77, 7)
Saved tracks: (441, 8)


In [9]:
albums_df.head()


,album_id,album_name,album_type,release_date,total_tracks,spotify_url,image_url
0,6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,album,2026-03-27,9,https://open.spotify.com/album/6iPjmGZeonxBZ9r...,https://i.scdn.co/image/ab67616d0000b273dd898e...
1,3ukkRHDHbN8tNRPKsGZR1h,ARIRANG,album,2026-03-20,14,https://open.spotify.com/album/3ukkRHDHbN8tNRP...,https://i.scdn.co/image/ab67616d0000b273dfa17f...
2,7xLhqoM0ilHRKUg2irVSDI,PERMISSION TO DANCE ON STAGE - LIVE,album,2025-07-18,22,https://open.spotify.com/album/7xLhqoM0ilHRKUg...,https://i.scdn.co/image/ab67616d0000b273b79d46...
3,6al2VdKbb6FIz9d7lU7WRB,Proof,album,2022-06-10,35,https://open.spotify.com/album/6al2VdKbb6FIz9d...,https://i.scdn.co/image/ab67616d0000b27317db30...
4,6nYfHQnvkvOTNHnOhDT3sr,BE,album,2020-11-20,8,https://open.spotify.com/album/6nYfHQnvkvOTNHn...,https://i.scdn.co/image/ab67616d0000b273c07d5d...


In [10]:
tracks_df.head()

,track_id,track_name,album_id,album_name,track_number,duration_ms,explicit,spotify_url
0,6nt3AoYjkaqXMZhypTBky1,SWIM,6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,1,159007,False,https://open.spotify.com/track/6nt3AoYjkaqXMZh...
1,7EytKcb3klVPpN5IW1sj1Y,SWIM with RM (Chill Hip Hop Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,2,163226,False,https://open.spotify.com/track/7EytKcb3klVPpN5...
2,5dZLsPskKzph16LWo31uxL,SWIM with Jin (Alternative Rock Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,3,193813,False,https://open.spotify.com/track/5dZLsPskKzph16L...
3,5AL5OrvyIMPqKjl9iw3xO5,SWIM with SUGA (Melodic Techno Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,4,219573,False,https://open.spotify.com/track/5AL5OrvyIMPqKjl...
4,3MCJY7lXCHa0UNIjsAucaJ,SWIM with j-hope (Afrobeat Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,5,153133,False,https://open.spotify.com/track/3MCJY7lXCHa0UNI...
